In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

In [2]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV

In [3]:
import plotly.express as px
import plotly.graph_objs as go
import sklearn

# <center> Подбор гиперпараметров </center>

In [4]:
df = pd.read_csv("https://raw.githubusercontent.com/katarina74/ml_lessons/main/lesson_2/data/techparams_train.csv")
X = df.drop(["target", "index"], axis=1)
y = df[["target"]]

ordinal = ['back-suspension', 'cylinders-order', 'engine-feeding', 'configurations_seats']
numerical = ['battery-capacity', 'charge-time', 'compression', 'consumption-mixed', 'cylinders-value', 'luxury',
             'max-speed', 'power-electro-kw','valves','weight','configurations_auto-premiere','configurations_back-wheel-base','configurations_tank-volume']
catigorial = ['engine-start','supergen_year-stop','engine-stop','engine-type','gear-type', 'supply-system', 'valvetrain', 'configurations_front-brake', 'configurations_safety-rating','models_country-from',
              'models_group', 'models_light-and-commercial','models_male']

#X_crop = X.drop(catigorial, axis=1)

train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
train_X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28974 entries, 22607 to 15795
Data columns (total 30 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   back-suspension                 28974 non-null  int64  
 1   battery-capacity                28974 non-null  float64
 2   charge-time                     28974 non-null  int64  
 3   compression                     28974 non-null  float64
 4   consumption-mixed               28974 non-null  float64
 5   cylinders-order                 28974 non-null  int64  
 6   cylinders-value                 28974 non-null  int64  
 7   engine-feeding                  28974 non-null  int64  
 8   engine-start                    28974 non-null  int64  
 9   engine-stop                     28974 non-null  int64  
 10  engine-type                     28974 non-null  int64  
 11  gear-type                       28974 non-null  int64  
 12  luxury                          2

In [ ]:
train_X.head()

,back-suspension,battery-capacity,charge-time,compression,consumption-mixed,cylinders-order,cylinders-value,engine-feeding,engine-start,engine-stop,...,configurations_back-wheel-base,configurations_front-brake,configurations_safety-rating,configurations_seats,configurations_tank-volume,supergen_year-stop,models_country-from,models_group,models_light-and-commercial,models_male
22607,3,-1.0,11636,9.5,-1.0,0,4,4,1997,2000,...,1415.0,4,2,13,50.0,2000.0,16,3,0,1
36531,8,-1.0,31764,9.0,-1.0,0,4,0,1990,1993,...,1310.0,1,2,13,48.0,1996.0,23,3,0,0
24048,8,-1.0,2773,8.8,6.8,0,4,4,2009,2012,...,1558.0,4,1,13,56.0,2012.0,10,3,0,1
34819,3,-1.0,52039,14.0,5.3,0,4,5,2015,2017,...,1590.0,4,2,13,56.0,2017.0,16,3,0,0
1328,3,-1.0,20689,18.0,7.5,0,5,5,2004,2005,...,1560.0,4,2,13,70.0,2009.0,28,3,0,1


In [ ]:
reg = LinearRegression().fit(train_X, train_y)

In [ ]:
pred_test = reg.predict(test_X)

In [ ]:
mean_absolute_percentage_error(test_y, pred_test)

0.044973108044071584

### Кросс валидация

In [ ]:
model = LinearRegression()

In [11]:
scores = cross_val_score(model, train_X, train_y, cv = 10, scoring = 'neg_mean_absolute_percentage_error', verbose = 1)

[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed:    0.4s finished


In [12]:
print(scores)
(scores.mean(), scores.var())

[-0.0451511  -0.0447811  -0.04466816 -0.04293119 -0.0432839  -0.04401972
 -0.04536235 -0.0451736  -0.0437018  -0.04388562]


(np.float64(-0.0442958545871149), np.float64(6.497167653032127e-07))

In [ ]:
kf = KFold(n_splits = 10, shuffle = True, random_state = 42)

In [14]:
fold_n = 1
for train_index, test_index in kf.split(train_X, train_y):
    model = LinearRegression()
    model.fit(train_X.iloc[train_index], train_y.iloc[train_index])
    print(fold_n, mean_absolute_percentage_error(train_y.iloc[test_index], model.predict(train_X.iloc[test_index])[:,0]))
    

1 0.04435009564656798
1 0.045874997817309915
1 0.044660013035448996
1 0.04402936701266291
1 0.04460401546854753


1 0.04400784942994144
1 0.04376047393111144
1 0.04405575460900875
1 0.04415463029449327
1 0.043354504564340555


### Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
numerical_transformer = StandardScaler()

In [ ]:
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical),
        ('cat', categorical_transformer, catigorial)])

In [ ]:
train_X

,back-suspension,battery-capacity,charge-time,compression,consumption-mixed,cylinders-order,cylinders-value,engine-feeding,engine-start,engine-stop,...,configurations_back-wheel-base,configurations_front-brake,configurations_safety-rating,configurations_seats,configurations_tank-volume,supergen_year-stop,models_country-from,models_group,models_light-and-commercial,models_male
22607,3,-1.0,11636,9.5,-1.0,0,4,4,1997,2000,...,1415.0,4,2,13,50.0,2000.0,16,3,0,1
36531,8,-1.0,31764,9.0,-1.0,0,4,0,1990,1993,...,1310.0,1,2,13,48.0,1996.0,23,3,0,0
24048,8,-1.0,2773,8.8,6.8,0,4,4,2009,2012,...,1558.0,4,1,13,56.0,2012.0,10,3,0,1
34819,3,-1.0,52039,14.0,5.3,0,4,5,2015,2017,...,1590.0,4,2,13,56.0,2017.0,16,3,0,0
1328,3,-1.0,20689,18.0,7.5,0,5,5,2004,2005,...,1560.0,4,2,13,70.0,2009.0,28,3,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6265,6,-1.0,42729,16.5,4.7,0,4,5,2017,0,...,1630.0,4,2,13,68.0,-1.0,10,3,0,1
11284,0,-1.0,46093,8.1,-1.0,3,7,0,1987,1990,...,1486.0,4,2,13,82.0,1996.0,34,3,0,1
38158,3,-1.0,48819,10.0,7.6,0,4,2,2018,0,...,1636.0,4,2,13,62.0,-1.0,26,3,0,1
860,10,-1.0,9346,22.4,-1.0,0,4,6,1992,1995,...,1460.0,1,2,13,50.0,1995.0,16,3,0,1


In [ ]:
reg = LinearRegression()

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', reg)])

In [22]:
model.fit(train_X, train_y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['battery-capacity',
                                                   'charge-time', 'compression',
                                                   'consumption-mixed',
                                                   'cylinders-value', 'luxury',
                                                   'max-speed',
                                                   'power-electro-kw', 'valves',
                                                   'weight',
                                                   'configurations_auto-premiere',
                                                   'configurations_back-wheel-base',
                                                   'configurations_tank-volume']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['engine-start',
                                                   'supergen_year-stop',
                                                   'engine-stop', 'engine-type',
                                                   'gear-type', 'supply-system',
                                                   'valvetrain',
                                                   'configurations_front-brake',
                                                   'configurations_safety-rating',
                                                   'models_country-from',
                                                   'models_group',
                                                   'models_light-and-commercial',
                                                   'models_male'])])),
                ('regressor', LinearRegression())])

In [ ]:
pred_test = model.predict(test_X)

In [ ]:
mean_absolute_percentage_error(test_y, pred_test)

0.04379747435611562

In [25]:
scores = cross_val_score(model, train_X, train_y, cv = 10, scoring = 'neg_mean_absolute_percentage_error', verbose = 1)


[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed:   31.7s finished


In [26]:
print(scores)
(scores.mean(), scores.var())

[-0.04279724 -0.0427149  -0.04368984 -0.04260933 -0.04257747 -0.04289418
 -0.04496673 -0.04451464 -0.04305201 -0.0436945 ]


(np.float64(-0.04335108274774393), np.float64(6.355220966854175e-07))

### Подбор параметров модели

In [ ]:
reg = ElasticNet()
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', reg)])

In [ ]:
parameters = [
    {'regressor__alpha': [0.01, 0.5, 1]},
    {'regressor__l1_ratio': [0.01, 0.5, 1]}]

In [ ]:
kf = KFold(n_splits = 10, shuffle = True, random_state = 42)

In [30]:
model_cv = GridSearchCV(model, parameters, cv = kf, verbose = 2, scoring = 'neg_mean_absolute_percentage_error')

In [31]:
model_cv.fit(train_X, train_y)

Fitting 10 folds for each of 6 candidates, totalling 60 fits
[CV] END ..............................regressor__alpha=0.01; total time=  12.9s
[CV] END ..............................regressor__alpha=0.01; total time=  13.3s
[CV] END ..............................regressor__alpha=0.01; total time=  13.2s
[CV] END ..............................regressor__alpha=0.01; total time=  13.8s
[CV] END ..............................regressor__alpha=0.01; total time=  14.2s
[CV] END ..............................regressor__alpha=0.01; total time=  13.2s
[CV] END ..............................regressor__alpha=0.01; total time=  12.5s
[CV] END ..............................regressor__alpha=0.01; total time=  13.0s
[CV] END ..............................regressor__alpha=0.01; total time=  12.7s
[CV] END ..............................regressor__alpha=0.01; total time=  12.6s
[CV] END ...............................regressor__alpha=0.5; total time=   0.7s
[CV] END ...............................regresso

GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['battery-capacity',
                                                                          'charge-time',
                                                                          'compression',
                                                                          'consumption-mixed',
                                                                          'cylinders-value',
                                                                          'luxury',
                                                                          'max-speed',
                                                                          'power-electro-kw',
                                                                          'valves',
                                                                          'weight',
                                                                          'configurations_auto-premiere',
                                                                          'configuratio...
                                                                          'gear-type',
                                                                          'supply-system',
                                                                          'valvetrain',
                                                                          'configurations_front-brake',
                                                                          'configurations_safety-rating',
                                                                          'models_country-from',
                                                                          'models_group',
                                                                          'models_light-and-commercial',
                                                                          'models_male'])])),
                                       ('regressor', ElasticNet())]),
             param_grid=[{'regressor__alpha': [0.01, 0.5, 1]},
                         {'regressor__l1_ratio': [0.01, 0.5, 1]}],
             scoring='neg_mean_absolute_percentage_error', verbose=2)

In [32]:
results = model_cv.cv_results_
results.keys()

dict_keys(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time', 'param_regressor__alpha', 'param_regressor__l1_ratio', 'params', 'split0_test_score', 'split1_test_score', 'split2_test_score', 'split3_test_score', 'split4_test_score', 'split5_test_score', 'split6_test_score', 'split7_test_score', 'split8_test_score', 'split9_test_score', 'mean_test_score', 'std_test_score', 'rank_test_score'])

In [33]:
go.Figure(data = [
    go.Bar(y = results['mean_fit_time'], hovertext = [str(i) for i in results['params']]),
    go.Bar(y = -results['mean_test_score'])
                 ])

In [34]:
model_cv.best_params_

{'regressor__alpha': 0.01}

In [35]:
model_cv.best_score_

np.float64(-0.043459437509543675)

In [36]:
pred_test = model_cv.predict(test_X)

In [37]:
pred_test = model_cv.best_estimator_.predict(test_X)

In [38]:
mean_absolute_percentage_error(test_y, pred_test)

0.04399098501429684

### Optune

In [40]:
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 4.4 MB/s eta 0:00:00a 0:00:01


In [41]:
reg = ElasticNet()
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', reg)])

In [42]:
parameters = [
    {'regressor__alpha': [0.01, 0.5, 1]},
    {'regressor__l1_ratio': [0.01, 0.5, 1]}]

In [43]:
param_distributions = {
        "regressor__alpha": optuna.distributions.FloatDistribution(0.01, 1),
        "regressor__l1_ratio": optuna.distributions.FloatDistribution(0.01,1),
    }

In [44]:
?optuna.integration.OptunaSearchCV

Object `optuna.integration.OptunaSearchCV` not found.


In [46]:
!pip install optuna-integration[sklearn]
optuna_search = optuna.integration.OptunaSearchCV(
        model, param_distributions, n_trials=10, timeout=600, verbose=2, scoring = 'neg_mean_absolute_percentage_error'
    )

optuna_search.fit(train_X, train_y)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 2.0 MB/s eta 0:00:00 0:00:01


/tmp/ipykernel_1114/818243538.py:2: ExperimentalWarning:

OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.

[I 2026-05-04 13:54:14,761] A new study created in memory with name: no-name-49b00a59-306b-48c9-8dcb-b986c6894af2
INFO:optuna_integration.sklearn.sklearn:Searching the best hyperparameters using 28974 samples...
[I 2026-05-04 13:54:52,985] Trial 0 finished with value: -0.043575830374827526 and parameters: {'regressor__alpha': 0.011755306512167085, 'regressor__l1_ratio': 0.2719440250451275}. Best is trial 0 with value: -0.043575830374827526.
[I 2026-05-04 13:54:56,348] Trial 1 finished with value: -0.04490805542451172 and parameters: {'regressor__alpha': 0.314899968694659, 'regressor__l1_ratio': 0.5954673738469047}. Best is trial 0 with value: -0.043575830374827526.
[I 2026-05-04 13:54:58,447] Trial 2 finished with value: -0.04764561259251259 and parameters: {'regressor__alpha': 0.7881846443184801, 'regressor__l1_ratio': 0.2594204808

OptunaSearchCV(estimator=Pipeline(steps=[('preprocessor',
                                          ColumnTransformer(transformers=[('num',
                                                                           StandardScaler(),
                                                                           ['battery-capacity',
                                                                            'charge-time',
                                                                            'compression',
                                                                            'consumption-mixed',
                                                                            'cylinders-value',
                                                                            'luxury',
                                                                            'max-speed',
                                                                            'power-electro-kw',
                                                                            'valves',
                                                                            'weight',
                                                                            'configurations_auto-premiere',
                                                                            'configurations_back-wheel-base',
                                                                            'configurations_tank-volume']...
                                                                            'models_country-from',
                                                                            'models_group',
                                                                            'models_light-and-commercial',
                                                                            'models_male'])])),
                                         ('regressor', ElasticNet())]),
               n_jobs=1,
               param_distributions={'regressor__alpha': FloatDistribution(high=1.0, log=False, low=0.01, step=None),
                                    'regressor__l1_ratio': FloatDistribution(high=1.0, log=False, low=0.01, step=None)},
               scoring='neg_mean_absolute_percentage_error', timeout=600,
               verbose=2)

In [47]:
print("Best trial:")
trial = optuna_search.study_.best_trial
print("  Value: ", trial.value)

Best trial:
  Value:  -0.043575830374827526


In [48]:
print("  Params: ")
for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

  Params: 
    regressor__alpha: 0.011755306512167085
    regressor__l1_ratio: 0.2719440250451275


In [49]:
pred_test_opt = optuna_search.predict(test_X)

In [50]:
mean_absolute_percentage_error(test_y, pred_test_opt)

0.04410431452889491

**Задание**

Изучить https://optuna.readthedocs.io/en/stable/tutorial/index.html.
Рализовать выбор лучшей модели с помощью библиотеки optune.
Оптимизатор должен за 100 шагов:
- выбрать лучший способ стандартизации количественных признаков;
- выбрать лучшую модель линейной регрессии;
- подобрать гиперпараметры модели.

Вывести подобранные величины.

In [51]:
regression_name = trial.suggest_categorical('regression', ['Lasso', 'Ridge', 'ElasticNet'])

ValueError: The value of the parameter 'regression' is not found. Please set it at the construction of the FrozenTrial object.

In [54]:
def objective(trial):


    scale_name = trial.suggest_categorical('scale', ['StandardScaler', 'RobustScaler', 'MinMaxScaler'])

    if scale_name == 'StandardScaler':
        scaler = StandardScaler()
    elif scale_name == 'RobustScaler':
        scaler = RobustScaler()
    else:
        scaler = MinMaxScaler()
    
    x, y = scaler.fit_transform(train_X), train_y

    regression_name = trial.suggest_categorical('regression', ['Lasso', 'Ridge', 'ElasticNet', 'LinearRegression'])
    if regression_name == "Lasso":
        alpha = trial.suggest_float("alpha", 0, 1)
        regression_obj = Lasso(alpha=alpha)
    elif regression_name == "Ridge":
        alpha = trial.suggest_float("alpha", 0, 1)
        regression_obj = Ridge(alpha=alpha)
    elif regression_name == 'LinearRegression':
        regression_obj = LinearRegression()
    else:
        alpha = trial.suggest_float("alpha", 0, 1)
        l1_ratio = trial.suggest_float("l1_ratio", 0, 1)
        regression_obj = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)

    score = sklearn.model_selection.cross_val_score(regression_obj, x, y, n_jobs=-1, cv=5, scoring='r2')
    accuracy = score.mean()
    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100)
    print(study.best_trial)

[I 2026-05-04 13:57:29,435] A new study created in memory with name: no-name-0295e0a8-23da-4794-a44f-40872f2c0238


[I 2026-05-04 13:57:29,791] Trial 0 finished with value: 0.4540477568161424 and parameters: {'scale': 'StandardScaler', 'regression': 'Lasso', 'alpha': 0.5466274518795519}. Best is trial 0 with value: 0.4540477568161424.
[I 2026-05-04 13:57:30,425] Trial 1 finished with value: 0.4540855548813907 and parameters: {'scale': 'StandardScaler', 'regression': 'Lasso', 'alpha': 0.22823927173433212}. Best is trial 1 with value: 0.4540855548813907.
[I 2026-05-04 13:57:30,878] Trial 2 finished with value: 0.4520050488522429 and parameters: {'scale': 'RobustScaler', 'regression': 'Lasso', 'alpha': 0.9963013907055127}. Best is trial 1 with value: 0.4540855548813907.
[I 2026-05-04 13:57:31,112] Trial 3 finished with value: 0.06663525694417624 and parameters: {'scale': 'MinMaxScaler', 'regression': 'ElasticNet', 'alpha': 0.898383517278058, 'l1_ratio': 0.09621372122018634}. Best is trial 1 with value: 0.4540855548813907.
[I 2026-05-04 13:57:31,223] Trial 4 finished with value: 0.45415419754499686 and 

FrozenTrial(number=51, state=<TrialState.COMPLETE: 1>, values=[0.4541542646380693], datetime_start=datetime.datetime(2026, 5, 4, 13, 57, 43, 344914), datetime_complete=datetime.datetime(2026, 5, 4, 13, 57, 43, 583444), params={'scale': 'MinMaxScaler', 'regression': 'Ridge', 'alpha': 0.41980803014031187}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'scale': CategoricalDistribution(choices=('StandardScaler', 'RobustScaler', 'MinMaxScaler')), 'regression': CategoricalDistribution(choices=('Lasso', 'Ridge', 'ElasticNet', 'LinearRegression')), 'alpha': FloatDistribution(high=1.0, log=False, low=0.0, step=None)}, trial_id=51, value=None)


In [53]:
# value: 0.4538629185314214 and parameters: {'scale': 'StandardScaler', 'regression': 'Ridge', 'alpha': 3.455322262964176}